# 01 · Data Loading & SQL Exploration

This notebook covers the first phase of the Barcelona Housing Affordability Analysis:

- Loading the three raw datasets (cadastral values, housing surface, household income) for the period 2018–2022
- Exploring the structure and content of each dataset
- Building a relational SQLite database
- Running SQL queries to validate joins and extract initial insights

**Data sources:** Open Data Barcelona (Ajuntament de Barcelona)  
**Period:** 2018–2022  
**Granularity:** Neighbourhood (barri) and district (districte)

In [2]:
import pandas as pd
import sqlite3
import os
from pathlib import Path

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
pd.set_option('display.float_format', '{:,.2f}'.format)

print("Libraries loaded successfully")

Libraries loaded successfully


In [4]:
# Project root (one level up from notebooks/)
ROOT = Path().resolve().parent

RAW_CADASTRAL = ROOT / "data" / "raw" / "cadastral_values"
RAW_SURFACE   = ROOT / "data" / "raw" / "housing_surface"
RAW_INCOME    = ROOT / "data" / "raw" / "household_income"
PROCESSED     = ROOT / "data" / "processed"

PROCESSED.mkdir(parents=True, exist_ok=True)

# Verify files are accessible
for folder in [RAW_CADASTRAL, RAW_SURFACE, RAW_INCOME]:
    files = sorted(folder.glob("*.csv"))
    print(f"\n{folder.name}/")
    for f in files:
        print(f"  {f.name}")


cadastral_values/
  2018_loc_hab_valors.csv
  2019_loc_hab_valors.csv
  2020_loc_hab_valors.csv
  2021_loc_hab_valors.csv
  2022_loc_hab_valors.csv

housing_surface/
  2018_loc_hab_sup.csv
  2019_loc_hab_sup.csv
  2020_loc_hab_sup.csv
  2021_loc_hab_sup.csv
  2022_loc_hab_sup.csv

household_income/
  2018_renda_disponible_llars_per_persona.csv
  2019_renda_disponible_llars_per_persona.csv
  2020_renda_disponible_llars_per_persona.csv
  2021_renda_disponible_llars_per_persona.csv
  2022_renda_disponible_llars_per_persona.csv


## 1. Raw Data Exploration

Before loading all files, we inspect one file per dataset to understand column names, data types, encoding and general structure.

In [7]:
YEARS = [2018, 2019, 2020, 2021, 2022]

# Load one sample file per dataset
sample_cadastral = pd.read_csv(RAW_CADASTRAL / "2022_loc_hab_valors.csv", sep=None, engine='python', nrows=5)
sample_surface   = pd.read_csv(RAW_SURFACE   / "2022_loc_hab_sup.csv",    sep=None, engine='python', nrows=5)
sample_income    = pd.read_csv(RAW_INCOME    / "2022_renda_disponible_llars_per_persona.csv", sep=None, engine='python', nrows=5)

print("=== CADASTRAL VALUES ===")
print(f"Shape preview: {sample_cadastral.shape}")
print(sample_cadastral.head())

print("\n=== HOUSING SURFACE ===")
print(f"Shape preview: {sample_surface.shape}")
print(sample_surface.head())

print("\n=== HOUSEHOLD INCOME ===")
print(f"Shape preview: {sample_income.shape}")
print(sample_income.head())

=== CADASTRAL VALUES ===
Shape preview: (5, 8)
    Any  Codi_districte Nom_districte  Codi_barri Nom_barri  Seccio_censal  \
0  2022               1  Ciutat Vella           1  el Raval              1   
1  2022               1  Ciutat Vella           1  el Raval              2   
2  2022               1  Ciutat Vella           1  el Raval              3   
3  2022               1  Ciutat Vella           1  el Raval              4   
4  2022               1  Ciutat Vella           1  el Raval              5   

               Desc_valors         Valors  
0  Valor_cadastral_total_€  66,331,900.15  
1  Valor_cadastral_total_€  52,258,919.83  
2  Valor_cadastral_total_€  72,381,494.27  
3  Valor_cadastral_total_€  97,473,125.23  
4  Valor_cadastral_total_€ 100,662,701.86  

=== HOUSING SURFACE ===
Shape preview: (5, 8)
    Any  Codi_districte Nom_districte  Codi_barri Nom_barri  Seccio_censal  \
0  2022               1  Ciutat Vella           1  el Raval              1   
1  2022          

## 2. Data Loading & Column Standardisation

We load all years for each dataset, concatenate them into a single DataFrame per source, rename columns from Catalan to English, and apply basic type casting.

In [10]:
# Column renaming maps
COLS_CADASTRAL = {
    'Any': 'year',
    'Codi_districte': 'district_id',
    'Nom_districte': 'district_name',
    'Codi_barri': 'neighbourhood_id',
    'Nom_barri': 'neighbourhood_name',
    'Seccio_censal': 'census_section',
    'Desc_valors': 'value_type',
    'Valors': 'value'
}

COLS_SURFACE = {
    'Any': 'year',
    'Codi_districte': 'district_id',
    'Nom_districte': 'district_name',
    'Codi_barri': 'neighbourhood_id',
    'Nom_barri': 'neighbourhood_name',
    'Seccio_censal': 'census_section',
    'Desc_sup': 'surface_range',
    'Nombre': 'housing_count'
}

COLS_INCOME = {
    'Any': 'year',
    'Codi_Districte': 'district_id',
    'Nom_Districte': 'district_name',
    'Codi_Barri': 'neighbourhood_id',
    'Nom_Barri': 'neighbourhood_name',
    'Seccio_Censal': 'census_section',
    'Import_Euros': 'income_per_capita'
}

def load_dataset(folder, filename_pattern, col_map):
    frames = []
    for year in YEARS:
        filepath = folder / filename_pattern.format(year=year)
        df = pd.read_csv(filepath, sep=None, engine='python')
        df = df.rename(columns=col_map)
        frames.append(df)
    return pd.concat(frames, ignore_index=True)

df_cadastral = load_dataset(RAW_CADASTRAL, "{year}_loc_hab_valors.csv",                        COLS_CADASTRAL)
df_surface   = load_dataset(RAW_SURFACE,   "{year}_loc_hab_sup.csv",                           COLS_SURFACE)
df_income    = load_dataset(RAW_INCOME,    "{year}_renda_disponible_llars_per_persona.csv",    COLS_INCOME)

print(f"Cadastral values : {df_cadastral.shape}")
print(f"Housing surface  : {df_surface.shape}")
print(f"Household income : {df_income.shape}")

Cadastral values : (32040, 8)
Housing surface  : (32314, 8)
Household income : (5340, 7)


In [12]:
# Fix 'value' column in cadastral (string with commas → float)
df_cadastral['value'] = (
    df_cadastral['value']
    .astype(str)
    .str.replace(',', '', regex=False)
    .astype(float)
)

# Ensure correct types across all datasets
for df in [df_cadastral, df_surface, df_income]:
    df['year']             = df['year'].astype(int)
    df['district_id']      = df['district_id'].astype(int)
    df['neighbourhood_id'] = df['neighbourhood_id'].astype(int)
    df['census_section']   = df['census_section'].astype(int)

df_surface['housing_count']      = df_surface['housing_count'].astype(int)
df_income['income_per_capita']   = df_income['income_per_capita'].astype(float)

print("=== CADASTRAL VALUES ===")
print(df_cadastral.dtypes)
print(f"\nUnique value_type categories:\n{df_cadastral['value_type'].unique()}")

print("\n=== HOUSING SURFACE ===")
print(df_surface.dtypes)
print(f"\nUnique surface_range categories:\n{df_surface['surface_range'].unique()}")

print("\n=== HOUSEHOLD INCOME ===")
print(df_income.dtypes)

=== CADASTRAL VALUES ===
year                    int32
district_id             int32
district_name          object
neighbourhood_id        int32
neighbourhood_name     object
census_section          int32
value_type             object
value                 float64
dtype: object

Unique value_type categories:
['Valor_cadastral_total_€' 'Valor_sòl_total_€'
 'Valor_construcció_més_serveis_total_€' 'Valor_cadastral_unitari_€/m2'
 'Valor_sòl_unitari_€/m2' 'Valor_construcció_més_serveis_unitari_€/m2']

=== HOUSING SURFACE ===
year                   int32
district_id            int32
district_name         object
neighbourhood_id       int32
neighbourhood_name    object
census_section         int32
surface_range         object
housing_count          int32
dtype: object

Unique surface_range categories:
['Fins a 30 m2' '31- 60 m2' '61- 90 m2' '91- 120 m2' '121- 150 m2'
 '151- 210 m2' '211- 250 m2' 'Més de 250 m2']

=== HOUSEHOLD INCOME ===
year                    int32
district_id             i

## 3. Value Standardisation

Translating remaining Catalan category values to English and standardising `value_type` labels.

In [15]:
# Translate surface_range categories
SURFACE_RANGE_MAP = {
    'Fins a 30 m2'  : 'Up to 30 m2',
    '31- 60 m2'     : '31-60 m2',
    '61- 90 m2'     : '61-90 m2',
    '91- 120 m2'    : '91-120 m2',
    '121- 150 m2'   : '121-150 m2',
    '151- 210 m2'   : '151-210 m2',
    '211- 250 m2'   : '211-250 m2',
    'Més de 250 m2' : 'Over 250 m2'
}

# Translate value_type categories
VALUE_TYPE_MAP = {
    'Valor_cadastral_total_€'                      : 'total_cadastral_value_eur',
    'Valor_sòl_total_€'                            : 'total_land_value_eur',
    'Valor_construcció_més_serveis_total_€'        : 'total_construction_value_eur',
    'Valor_cadastral_unitari_€/m2'                 : 'unit_cadastral_value_eur_m2',
    'Valor_sòl_unitari_€/m2'                       : 'unit_land_value_eur_m2',
    'Valor_construcció_més_serveis_unitari_€/m2'   : 'unit_construction_value_eur_m2'
}

df_surface['surface_range']  = df_surface['surface_range'].map(SURFACE_RANGE_MAP)
df_cadastral['value_type']   = df_cadastral['value_type'].map(VALUE_TYPE_MAP)

print("surface_range unique values:")
print(df_surface['surface_range'].unique())

print("\nvalue_type unique values:")
print(df_cadastral['value_type'].unique())

print("\nAny nulls introduced by mapping?")
print(f"  surface_range nulls : {df_surface['surface_range'].isna().sum()}")
print(f"  value_type nulls    : {df_cadastral['value_type'].isna().sum()}")

surface_range unique values:
['Up to 30 m2' '31-60 m2' '61-90 m2' '91-120 m2' '121-150 m2' '151-210 m2'
 '211-250 m2' 'Over 250 m2']

value_type unique values:
['total_cadastral_value_eur' 'total_land_value_eur'
 'total_construction_value_eur' 'unit_cadastral_value_eur_m2'
 'unit_land_value_eur_m2' 'unit_construction_value_eur_m2']

Any nulls introduced by mapping?
  surface_range nulls : 0
  value_type nulls    : 0


## 4. SQLite Database

We store the three cleaned DataFrames as tables in a local SQLite database. This allows us to query the data using standard SQL, demonstrating relational database skills including joins, aggregations and filtering.

**Tables:**
- `cadastral_values` — unit and total cadastral values per census section
- `housing_surface` — number of dwellings by surface range per census section
- `household_income` — estimated household income per capita per census section

In [18]:
DB_PATH = PROCESSED / "barcelona_housing.db"

conn = sqlite3.connect(DB_PATH)

df_cadastral.to_sql('cadastral_values', conn, if_exists='replace', index=False)
df_surface.to_sql('housing_surface',    conn, if_exists='replace', index=False)
df_income.to_sql('household_income',    conn, if_exists='replace', index=False)

# Verify tables were created
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
tables = cursor.fetchall()

print("Tables in database:")
for t in tables:
    print(f"  {t[0]}")

# Row counts
for table in ['cadastral_values', 'housing_surface', 'household_income']:
    cursor.execute(f"SELECT COUNT(*) FROM {table}")
    count = cursor.fetchone()[0]
    print(f"  {table}: {count:,} rows")

print(f"\nDatabase saved to: {DB_PATH}")

Tables in database:
  cadastral_values
  housing_surface
  household_income
  cadastral_values: 32,040 rows
  housing_surface: 32,314 rows
  household_income: 5,340 rows

Database saved to: C:\Users\oscar\Desktop\CV\PROYECTOS_GITHUB\barcelona-housing-affordability-analysis\data\processed\barcelona_housing.db


## 5. SQL Queries

We run a series of SQL queries directly on the database to validate the data and extract initial insights that will feed into the next notebooks.

In [21]:
def run_query(sql, conn):
    return pd.read_sql_query(sql, conn)

# Q1 — Years and districts available in each table
for table in ['cadastral_values', 'housing_surface', 'household_income']:
    df = run_query(f"""
        SELECT year, COUNT(DISTINCT district_id) AS districts,
               COUNT(DISTINCT neighbourhood_id)  AS neighbourhoods,
               COUNT(DISTINCT census_section)    AS census_sections
        FROM {table}
        GROUP BY year
        ORDER BY year
    """, conn)
    print(f"\n=== {table} ===")
    print(df.to_string(index=False))


=== cadastral_values ===
 year  districts  neighbourhoods  census_sections
 2018         10              73              181
 2019         10              73              181
 2020         10              73              181
 2021         10              73              181
 2022         10              73              181

=== housing_surface ===
 year  districts  neighbourhoods  census_sections
 2018         10              73              181
 2019         10              73              181
 2020         10              73              181
 2021         10              73              181
 2022         10              73              181

=== household_income ===
 year  districts  neighbourhoods  census_sections
 2018         10              73              181
 2019         10              73              181
 2020         10              73              181
 2021         10              73              181
 2022         10              73              181


In [23]:
# Q2 — Average unit cadastral value per m2 by district, comparing 2018 vs 2022
q2 = run_query("""
    SELECT
        district_name,
        ROUND(AVG(CASE WHEN year = 2018 THEN value END), 2) AS avg_value_2018,
        ROUND(AVG(CASE WHEN year = 2022 THEN value END), 2) AS avg_value_2022,
        ROUND(
            (AVG(CASE WHEN year = 2022 THEN value END) - 
             AVG(CASE WHEN year = 2018 THEN value END)) /
             AVG(CASE WHEN year = 2018 THEN value END) * 100
        , 2) AS pct_change
    FROM cadastral_values
    WHERE value_type = 'unit_cadastral_value_eur_m2'
    GROUP BY district_name
    ORDER BY pct_change DESC
""", conn)

print("=== Unit Cadastral Value (€/m²) by District: 2018 vs 2022 ===")
print(q2.to_string(index=False))

=== Unit Cadastral Value (€/m²) by District: 2018 vs 2022 ===
      district_name  avg_value_2018  avg_value_2022  pct_change
        Sant Andreu          787.21          792.41        0.66
         Sant Martí          833.30          836.78        0.42
           Eixample        1,166.92        1,170.65        0.32
       Ciutat Vella        1,016.63        1,019.66        0.30
     Sants-Montjuïc          842.32          844.75        0.29
             Gràcia        1,053.35        1,056.01        0.25
     Horta-Guinardó          725.37          726.93        0.22
         Nou Barris          630.05          631.02        0.15
Sarrià-Sant Gervasi        1,379.28        1,380.55        0.09
          Les Corts        1,252.57        1,253.37        0.06


In [25]:
# Q3 — Average household income per capita by district, comparing 2018 vs 2022
q3 = run_query("""
    SELECT
        district_name,
        ROUND(AVG(CASE WHEN year = 2018 THEN income_per_capita END), 2) AS avg_income_2018,
        ROUND(AVG(CASE WHEN year = 2022 THEN income_per_capita END), 2) AS avg_income_2022,
        ROUND(
            (AVG(CASE WHEN year = 2022 THEN income_per_capita END) -
             AVG(CASE WHEN year = 2018 THEN income_per_capita END)) /
             AVG(CASE WHEN year = 2018 THEN income_per_capita END) * 100
        , 2) AS pct_change
    FROM household_income
    GROUP BY district_name
    ORDER BY pct_change DESC
""", conn)

print("=== Household Income per Capita (€) by District: 2018 vs 2022 ===")
print(q3.to_string(index=False))

=== Household Income per Capita (€) by District: 2018 vs 2022 ===
     district_name  avg_income_2018  avg_income_2022  pct_change
      Ciutat Vella        15,312.85        17,033.31       11.24
        Sant Martí        19,369.33        21,350.67       10.23
    Sants-Montjuïc        18,596.86        20,463.53       10.04
        Nou Barris        15,629.74        17,093.24        9.36
       Sant Andreu        19,076.66        20,710.93        8.57
    Horta-Guinardó        19,153.27        20,736.37        8.27
            Gràcia        23,418.97        25,277.75        7.94
          Eixample        24,431.16        26,005.30        6.44
         Les Corts        28,376.46        29,393.79        3.59
Sarrià-St. Gervasi        34,599.13        35,155.53        1.61


In [27]:
# Q4 — Affordability ratio: unit cadastral value / income per capita by district
# Higher ratio = less affordable
q4 = run_query("""
    SELECT
        c.district_name,
        ROUND(AVG(c.value), 2)               AS avg_unit_value_eur_m2,
        ROUND(AVG(i.income_per_capita), 2)   AS avg_income_per_capita,
        ROUND(AVG(c.value) / AVG(i.income_per_capita), 4) AS affordability_ratio
    FROM cadastral_values c
    JOIN household_income i
        ON  c.year             = i.year
        AND c.district_id      = i.district_id
        AND c.neighbourhood_id = i.neighbourhood_id
        AND c.census_section   = i.census_section
    WHERE c.value_type = 'unit_cadastral_value_eur_m2'
      AND c.year = 2022
    GROUP BY c.district_name
    ORDER BY affordability_ratio DESC
""", conn)

print("=== Affordability Ratio by District (2022) ===")
print("(Higher ratio = less affordable)")
print(q4.to_string(index=False))

=== Affordability Ratio by District (2022) ===
(Higher ratio = less affordable)
      district_name  avg_unit_value_eur_m2  avg_income_per_capita  affordability_ratio
       Ciutat Vella               1,019.66              17,033.31                 0.06
           Eixample               1,170.65              26,005.30                 0.04
          Les Corts               1,253.37              29,393.79                 0.04
             Gràcia               1,056.01              25,277.75                 0.04
     Sants-Montjuïc                 844.75              20,463.53                 0.04
Sarrià-Sant Gervasi               1,380.55              35,155.53                 0.04
         Sant Martí                 836.78              21,350.67                 0.04
        Sant Andreu                 792.41              20,710.93                 0.04
         Nou Barris                 631.02              17,093.24                 0.04
     Horta-Guinardó                 726.93        

In [29]:
# Q5 — Share of dwellings under 60m2 by district (proxy for housing pressure)
q5 = run_query("""
    SELECT
        district_name,
        SUM(CASE WHEN surface_range IN ('Up to 30 m2', '31-60 m2') 
            THEN housing_count ELSE 0 END) AS small_dwellings,
        SUM(housing_count)                 AS total_dwellings,
        ROUND(
            100.0 * SUM(CASE WHEN surface_range IN ('Up to 30 m2', '31-60 m2')
                THEN housing_count ELSE 0 END) / SUM(housing_count)
        , 2) AS pct_small_dwellings
    FROM housing_surface
    WHERE year = 2022
    GROUP BY district_name
    ORDER BY pct_small_dwellings DESC
""", conn)

print("=== Share of Small Dwellings (≤60m²) by District (2022) ===")
print(q5.to_string(index=False))

=== Share of Small Dwellings (≤60m²) by District (2022) ===
      district_name  small_dwellings  total_dwellings  pct_small_dwellings
       Ciutat Vella            30524            57753                52.85
         Nou Barris            32963            76905                42.86
     Horta-Guinardó            32994            86103                38.32
     Sants-Montjuïc            29838            89034                33.51
             Gràcia            23205            70544                32.89
        Sant Andreu            21728            71377                30.44
         Sant Martí            29482           112519                26.20
           Eixample            27518           144334                19.07
Sarrià-Sant Gervasi            14769            77771                18.99
          Les Corts             7315            41217                17.75


In [31]:
# Q6 — Evolution of small dwellings share (≤60m²) per district across all years
q6 = run_query("""
    SELECT
        year,
        district_name,
        ROUND(
            100.0 * SUM(CASE WHEN surface_range IN ('Up to 30 m2', '31-60 m2')
                THEN housing_count ELSE 0 END) / SUM(housing_count)
        , 2) AS pct_small_dwellings
    FROM housing_surface
    GROUP BY year, district_name
    ORDER BY district_name, year
""", conn)

print("=== Evolution of Small Dwellings Share (≤60m²) by District ===")
print(q6.pivot(index='district_name', columns='year', 
               values='pct_small_dwellings').to_string())

=== Evolution of Small Dwellings Share (≤60m²) by District ===
year                 2018  2019  2020  2021  2022
district_name                                    
Ciutat Vella        52.87 52.86 52.80 52.88 52.85
Eixample            18.79 18.79 18.79 19.02 19.07
Gràcia              32.52 32.53 32.51 32.87 32.89
Horta-Guinardó      38.30 38.30 38.30 38.33 38.32
Les Corts           17.69 17.69 17.71 17.76 17.75
Nou Barris          43.07 43.02 42.97 42.98 42.86
Sant Andreu         30.69 30.69 30.43 30.31 30.44
Sant Martí          26.18 26.18 26.21 26.23 26.20
Sants-Montjuïc      33.08 33.08 33.10 33.48 33.51
Sarrià-Sant Gervasi 18.77 18.76 18.79 18.94 18.99


In [33]:
# Q7 — Master query joining all three tables at neighbourhood level (2022)
# This is the base query that will feed the affordability analysis in notebook 03
q7 = run_query("""
    SELECT
        c.year,
        c.district_id,
        c.district_name,
        c.neighbourhood_id,
        c.neighbourhood_name,
        ROUND(AVG(c.value), 2)                        AS avg_unit_cadastral_value,
        ROUND(AVG(i.income_per_capita), 2)            AS avg_income_per_capita,
        SUM(s.housing_count)                          AS total_dwellings,
        ROUND(100.0 * SUM(
            CASE WHEN s.surface_range IN ('Up to 30 m2', '31-60 m2')
            THEN s.housing_count ELSE 0 END
        ) / SUM(s.housing_count), 2)                  AS pct_small_dwellings
    FROM cadastral_values c
    JOIN household_income i
        ON  c.year             = i.year
        AND c.census_section   = i.census_section
        AND c.neighbourhood_id = i.neighbourhood_id
    JOIN housing_surface s
        ON  c.year             = s.year
        AND c.census_section   = s.census_section
        AND c.neighbourhood_id = s.neighbourhood_id
    WHERE c.value_type = 'unit_cadastral_value_eur_m2'
      AND c.year = 2022
    GROUP BY c.district_id, c.district_name, 
             c.neighbourhood_id, c.neighbourhood_name
    ORDER BY c.district_id, c.neighbourhood_id
""", conn)

print(f"=== Master Dataset: All Neighbourhoods (2022) ===")
print(f"Shape: {q7.shape}")
print(q7.head(10).to_string(index=False))

=== Master Dataset: All Neighbourhoods (2022) ===
Shape: (73, 9)
 year  district_id district_name  neighbourhood_id                    neighbourhood_name  avg_unit_cadastral_value  avg_income_per_capita  total_dwellings  pct_small_dwellings
 2022            1  Ciutat Vella                 1                              el Raval                    888.12              14,250.21            23413                55.95
 2022            1  Ciutat Vella                 2                              el Gòtic                  1,063.43              18,668.11            10499                33.73
 2022            1  Ciutat Vella                 3                        la Barceloneta                  1,228.09              18,089.74             9221                76.95
 2022            1  Ciutat Vella                 4 Sant Pere, Santa Caterina i la Ribera                  1,032.52              19,671.57            14620                46.43
 2022            2      Eixample                 5     

In [35]:
# Save master query result and full cleaned datasets to processed folder
q7.to_csv(PROCESSED / "neighbourhood_summary_2022.csv", index=False)
df_cadastral.to_csv(PROCESSED / "cadastral_values_clean.csv", index=False)
df_surface.to_csv(PROCESSED / "housing_surface_clean.csv", index=False)
df_income.to_csv(PROCESSED / "household_income_clean.csv", index=False)

print("Processed files saved:")
for f in sorted(PROCESSED.glob("*.csv")):
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:45s} {size_kb:>8.1f} KB")

Processed files saved:
  cadastral_values_clean.csv                      2509.9 KB
  household_income_clean.csv                       277.6 KB
  housing_surface_clean.csv                       1843.1 KB
  neighbourhood_summary_2022.csv                     5.1 KB


## 6. Summary & Key Findings

### Database
- Three tables loaded into SQLite: `cadastral_values` (32,040 rows), `housing_surface` (32,314 rows), `household_income` (5,340 rows)
- Perfect consistency across all tables: 10 districts, 73 neighbourhoods, 181 census sections per year (2018–2022)

### Key findings from SQL exploration

**Cadastral values are structurally stable (+0.06% to +0.66% over 2018–2022)**  
Cadastral values are fiscal estimates updated infrequently and sit well below market prices. They serve as a relative proxy for comparing districts, not as absolute market indicators. A market price correction factor will be applied in notebook 03.

**Income growth is unequal across districts**  
Lower-income districts (Ciutat Vella +11.2%, Sant Martí +10.2%) show higher relative income growth, but from a much lower base. Sarrià-Sant Gervasi grew only +1.6% but already had the highest income (€34,600 in 2018). Structural inequality persists.

**Small dwelling concentration reveals residential segregation**  
Ciutat Vella (52.9%) and Nou Barris (42.9%) have the highest share of dwellings under 60m², and also the lowest household incomes. Wealthier districts (Les Corts 17.8%, Sarrià-Sant Gervasi 19.0%) show the opposite pattern. La Barceloneta stands out with 76.9% small dwellings — the most extreme case in the city.

**This pattern is structurally persistent (2018–2022)**  
The share of small dwellings barely changed across any district over five years, confirming this is a structural feature of the housing stock, not a recent trend.

### Next steps
- `02_eda_and_cleaning.ipynb` — missing value analysis, outlier detection, feature engineering (affordability index, dwelling size score)
- `03_affordability_analysis.ipynb` — full affordability analysis with visualisations
- `04_clustering.ipynb` — K-Means clustering of neighbourhoods by affordability profile

In [40]:
conn.close()
print("Notebook 01 complete.")
print(f"\nOutputs saved to: data/processed/")
print(f"  - barcelona_housing.db")
print(f"  - cadastral_values_clean.csv")
print(f"  - housing_surface_clean.csv")
print(f"  - household_income_clean.csv")
print(f"  - neighbourhood_summary_2022.csv")

Notebook 01 complete.

Outputs saved to: data/processed/
  - barcelona_housing.db
  - cadastral_values_clean.csv
  - housing_surface_clean.csv
  - household_income_clean.csv
  - neighbourhood_summary_2022.csv
